In [31]:
import composable.records as rec

In [32]:
# Standard imports
import polars as pl
import polars.selectors as cs
import seaborn as sns
import numpy as np

# Preprocessing stuff
from sklearn.preprocessing import LabelEncoder

# Model selection stuff
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV

# Classic classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

# Classic regressors
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor

# Metrics to use on the test set
# metric(y_test, y_predict)
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score


# Trees and Forests in `sklearn`

In this notebook, we will cover the basics of classification, including:

1. The `DecisionTreeClassifier` and `RandomForestClassifier`.
2. Exploring tuning parameters for each model.
3. Performing a grid search.
4. Additional features of Random Forests.
5. Multiclass problems.

## Classification Example: Kyphosis Data Set

**Problem Statement:**
Given a DataSet of 81 patients who have undergone a spinal surgery for a deformation and the data if the condition recurred, Build a claddification Model to predict whether a patient being admitted for the surgery has chance for recurrence. This model will help the surgeons to plan appropriate level of treatment to prevent recurrence.

**Data Set Description :**
The kyphosis data frame has 81 rows and 4 columns. representing data on children who have had corrective spinal surgery

This data frame contains the following columns/Features:

1. *Kyphosis*: a factor with levels absent present indicating if a kyphosis (a type of deformation) was present after the operation.
2. *Age*: in months
3. *Number*: the number of vertebrae involved
4. *Start*: the number of the first (topmost) vertebra operated on.

**Research Question:** Can we predict whether if kyphosis will be present or absent after the surgery based on the age of the patient, number of vertebrae involved and the first vertebra operated on?

In [33]:
(kyphosis :=
 pl.read_csv('data/kyphosis.csv')
   .drop('rownames')
)

Kyphosis,Age,Number,Start
str,i64,i64,i64
"""absent""",71,3,5
"""absent""",158,3,14
"""present""",128,4,5
"""absent""",2,5,1
"""absent""",1,4,15
…,…,…,…
"""present""",157,3,13
"""absent""",26,7,13
"""absent""",120,2,13


In [34]:
(X_kyphosis :=
 kyphosis
 .drop('Kyphosis')
 .to_pandas()
)

,Age,Number,Start
0,71,3,5
1,158,3,14
2,128,4,5
3,2,5,1
4,1,4,15
...,...,...,...
76,157,3,13
77,26,7,13
78,120,2,13
79,42,7,6


In [35]:
(y_kyphosis :=
 kyphosis
 .get_column('Kyphosis')
 .to_numpy()
 .ravel()
)

array(['absent', 'absent', 'present', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'present', 'present', 'absent',
       'absent', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'present', 'present', 'absent',
       'present', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'present', 'absent', 'present', 'present', 'absent',
       'absent', 'absent', 'absent', 'present', 'absent', 'absent',
       'present', 'absent', 'absent', 'absent', 'present', 'absent',
       'absent', 'absent', 'absent', 'present', 'absent', 'absent',
       'present', 'present', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'absent', 'absent', 'absent',
       'absent', 'absent', 'absent', 'absent', 'present', 'absent',
       'absent', 'present', 'absent'], dtype=object)

## Topic 1 - Boosted Tree Classfiers

`sklearn` comes with a number of tree based boosted classifier, which we will explore in this section.

### Example 1 - AdaBoost for Classification

**AdaBoost (Adaptive Boosting)**: An ensemble learning method.
* Trains a series of weak learners sequentially.
* Each learner targets examples misclassified by previous ones.
* Misclassified examples receive higher weights.
* Combines weak models into a strong classifier.
* Often achieves high accuracy.

### Important AdaBoost Tuning Parameters

**Most important.** Here are some important parameters that we might tune:

* **`n_estimators`**: Number of weak learners (e.g., decision trees) to train.
* **`learning_rate`**: Shrinks the contribution of each weak learner. (0 to 1)

**Other parameters.** In addition, we could also tune the following:
* **`base_estimator`**: The type of weak learner used (default is DecisionTreeClassifier).

In [36]:
AdaBoostClassifier().get_params()

{'algorithm': 'deprecated',
 'estimator': None,
 'learning_rate': 1.0,
 'n_estimators': 50,
 'random_state': None}

In [37]:
(ada := AdaBoostClassifier()
)

,estimator,None
,n_estimators,50
,learning_rate,1.0
,algorithm,'deprecated'
,random_state,None


In [38]:
(folds := StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [39]:
(ada_grid :=
 {
      'learning_rate': np.linspace(0.1, 1.0, 10),
      'n_estimators': [25, 50, 100],
 }
)

{'learning_rate': array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ]),
 'n_estimators': [25, 50, 100]}

In [40]:
(ada_grid_search :=
 GridSearchCV(ada, ada_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)
)

,estimator,AdaBoostClassifier()
,param_grid,"{'learning_rate': array([0.1, 0....8, 0.9, 1. ]), 'n_estimators': [25, 50, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,estimator,None


In [41]:
ada_grid_search.fit(X_kyphosis, y_kyphosis)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,AdaBoostClassifier()
,param_grid,"{'learning_rate': array([0.1, 0....8, 0.9, 1. ]), 'n_estimators': [25, 50, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,estimator,None


In [42]:
ada_grid_search.best_params_, ada_grid_search.best_score_

({'learning_rate': np.float64(0.5), 'n_estimators': 50},
 np.float64(0.8759615384615385))

### Example 2 - Gradient Boost for Classification

### Gradient Boosting for Classification

**Gradient Boosting**: An ensemble technique building models sequentially that

*   Corrects errors of previous models.
*   Combines weak learners (e.g., decision trees).
*   Each new learner minimizes a loss function using gradient descent.
*   Forms a robust predictive model.
*   Highly effective for various tasks, including classification and regression.

### Important Gradient Boosting Tuning Parameters

**Most important tuning parameters** include
*   **`n_estimators`**: Number of boosting stages/weak learners.
*   **`learning_rate`**: Shrinks the contribution of each tree, preventing overfitting.
*   **`max_depth`**: Maximum depth of the individual regression estimators.

**Other tuning parameters.**
*   **`subsample`**: Fraction of samples used for fitting the individual base learners.
*   **`min_samples_split`**: Minimum number of samples required to split an internal node.
*   **`min_samples_leaf`**: Minimum number of samples required to be at a leaf node.

In [43]:
GradientBoostingClassifier().get_params()

{'ccp_alpha': 0.0,
 'criterion': 'friedman_mse',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'log_loss',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': None,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [44]:
(GB_classifier :=
 GradientBoostingClassifier()
)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [45]:
(GB_grid :=
 {
 'learning_rate': np.linspace(0.1, 1.0, 10),
 'n_estimators': [25, 50, 100],
 'max_depth': [3, 4, 5],
 }
)

{'learning_rate': array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ]),
 'n_estimators': [25, 50, 100],
 'max_depth': [3, 4, 5]}

In [46]:
(GB_grid_search :=
 GridSearchCV(GB_classifier, GB_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)
)

,estimator,GradientBoostingClassifier()
,param_grid,"{'learning_rate': array([0.1, 0....8, 0.9, 1. ]), 'max_depth': [3, 4, ...], 'n_estimators': [25, 50, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [47]:
GB_grid_search.fit(X_kyphosis, y_kyphosis)

Fitting 5 folds for each of 90 candidates, totalling 450 fits


,estimator,GradientBoostingClassifier()
,param_grid,"{'learning_rate': array([0.1, 0....8, 0.9, 1. ]), 'max_depth': [3, 4, ...], 'n_estimators': [25, 50, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [48]:
GB_grid_search.best_params_, GB_grid_search.best_score_

({'learning_rate': np.float64(0.7000000000000001),
  'max_depth': 3,
  'n_estimators': 100},
 np.float64(0.8176282051282051))

## Topic 2 - Histogram Gradient Boost

An new alternative to the gradient boost classifier
- Converts numeric features to a histogram
- Provides a substantial improvement in training for large data sets
- Has similar performance to the standard algorithm.

In [49]:
%%HTML

<iframe width="560" height="315" src="https://www.youtube.com/embed/5okmBJaE0kY?si=JQyXcydE0yrWtJzH" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

### Tuning the histogram gradient boost classifier

In addition to the standard parameters outlined above, you can also tune

- `class_weight` which can be set to `'balanced'` to account for class imbalance.
-   `max_bins` to cap the number of histogram bins.

In [50]:
(hist_GB :=
 HistGradientBoostingClassifier(class_weight='balanced')
)

,loss,'log_loss'
,learning_rate,0.1
,max_iter,100
,max_leaf_nodes,31
,max_depth,None
,min_samples_leaf,20
,l2_regularization,0.0
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'
,monotonic_cst,None


In [51]:
hist_GB.get_params()

{'categorical_features': 'from_dtype',
 'class_weight': 'balanced',
 'early_stopping': 'auto',
 'interaction_cst': None,
 'l2_regularization': 0.0,
 'learning_rate': 0.1,
 'loss': 'log_loss',
 'max_bins': 255,
 'max_depth': None,
 'max_features': 1.0,
 'max_iter': 100,
 'max_leaf_nodes': 31,
 'min_samples_leaf': 20,
 'monotonic_cst': None,
 'n_iter_no_change': 10,
 'random_state': None,
 'scoring': 'loss',
 'tol': 1e-07,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [52]:
(hist_GB_grid :=
    {
        'learning_rate': np.linspace(0.1, 1.0, 10),
        'max_iter': [25, 50, 100],
        'max_depth': [3, 4, 5],
        'max_bins': np.arange(2, 255, 75),
    }
)

{'learning_rate': array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ]),
 'max_iter': [25, 50, 100],
 'max_depth': [3, 4, 5],
 'max_bins': array([  2,  77, 152, 227])}

In [53]:
(
    hist_GB_grid_search :=
    GridSearchCV(hist_GB, hist_GB_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)
)

,estimator,HistGradientB...ht='balanced')
,param_grid,"{'learning_rate': array([0.1, 0....8, 0.9, 1. ]), 'max_bins': array([ 2, 77, 152, 227]), 'max_depth': [3, 4, ...], 'max_iter': [25, 50, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [54]:
hist_GB_grid_search.fit(X_kyphosis, y_kyphosis)

Fitting 5 folds for each of 360 candidates, totalling 1800 fits


,estimator,HistGradientB...ht='balanced')
,param_grid,"{'learning_rate': array([0.1, 0....8, 0.9, 1. ]), 'max_bins': array([ 2, 77, 152, 227]), 'max_depth': [3, 4, ...], 'max_iter': [25, 50, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [55]:
hist_GB_grid_search.best_params_, hist_GB_grid_search.best_score_

({'learning_rate': np.float64(0.1),
  'max_bins': np.int64(77),
  'max_depth': 3,
  'max_iter': 25},
 np.float64(0.8536858974358974))

### Example 4 - XGBoost for Classification

**XGBoost (eXtreme Gradient Boosting)**: An optimized distributed gradient boosting library.
*   Uses gradient boosting framework, similar to Gradient Boosting.
*   Known for its performance and speed.
*   Includes regularization (L1 and L2), parallel processing, tree pruning, and handling of sparse data.
*   Often wins machine learning competitions.
*   Highly scalable and accurate for classification and regression tasks.

### Important XGBoost Tuning Parameters

Here are some tuning parameters for XGBoost.
*   **`n_estimators`**: Number of boosting rounds or trees.
*   **`learning_rate`**: Step size shrinkage to prevent overfitting.
*   **`max_depth`**: Maximum depth of a tree.
*   **`subsample`**: Fraction of samples used for fitting the trees.
*   **`colsample_bytree`**: Fraction of features (columns) used when building each tree.
*   **`gamma`**: Minimum loss reduction required to make a further partition on a leaf node.
*   **`lambda` (L2)**: L2 regularization term on weights.
*   **`alpha` (L1)**: L1 regularization term on weights.

In [56]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


## <font color="red"> Exercise 1</font>

Watch the video posted above and answer the following:
1. How does the histogram varient differ from the regular algorithm?
2. What is the primary advantage of using the histogram varient?


<font color="orange">
The histogram variant (Histogram Gradient Boosting) first groups continuous feature values into a fixed number of bins, then searches splits on those bins.
The regular gradient boosting algorithm evaluates splits using the raw feature values (more exact, but more computationally expensive).

The primary advantage is much faster training (and usually lower memory use) on larger datasets, while often keeping similar predictive performance.
</font>

## <font color="red"> Exercise 2</font>

Use an LLM to investigate the following.
1. Get some advice about tuning XGBoost, and
2. Use an XGBoost classifier on the data presented above.


<font color="orange">


**XGBoost tuning advice (practical starting points):**


1. Start with **`max_depth`**, **`min_child_weight`**, and **`gamma`** to control model complexity.
2. Tune **`subsample`** and **`colsample_bytree`** (often 0.6 to 1.0) to reduce overfitting.
3. Use a **smaller `learning_rate`** with a **larger `n_estimators`** for better generalization.
4. Add regularization via **`reg_lambda`** (L2) and **`reg_alpha`** (L1) if overfitting appears.
5. Use **stratified cross-validation** and optimize with **ROC-AUC** for this binary classification task.
6. Keep search ranges moderate first, then do a second, narrower search around the best region.


For this kyphosis dataset, I use `GridSearchCV` with ROC-AUC and then evaluate the best model on a held-out test split.
</font>

In [64]:
from xgboost import XGBClassifier

# Encode target labels ("absent"/"present") to 0/1 for XGBoost
y_kyphosis_encoded = LabelEncoder().fit_transform(y_kyphosis)

In [65]:
# Train/test split for final evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X_kyphosis,
    y_kyphosis_encoded,
    test_size=0.2,
    stratify=y_kyphosis_encoded,
    random_state=42
)

In [70]:
(xgb_clf := XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
))

# Smaller grid for quick runs
(xgb_grid := {
    'n_estimators': [50, 100],
    'learning_rate': [0.05, 0.1],
    'max_depth': [2, 3, 4],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 0.25],
    'reg_lambda': [1, 5]
})

{'n_estimators': [50, 100],
 'learning_rate': [0.05, 0.1],
 'max_depth': [2, 3, 4],
 'subsample': [0.8, 1.0],
 'colsample_bytree': [0.8, 1.0],
 'gamma': [0, 0.25],
 'reg_lambda': [1, 5]}

In [71]:
(xgb_grid_search := GridSearchCV(
    estimator=xgb_clf,
    param_grid=xgb_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
))

xgb_grid_search.fit(X_train, y_train)

print('Best Params:', xgb_grid_search.best_params_)
print('Best CV ROC-AUC:', xgb_grid_search.best_score_)

Fitting 3 folds for each of 192 candidates, totalling 576 fits
Best Params: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 100, 'reg_lambda': 1, 'subsample': 0.8}
Best CV ROC-AUC: 0.8656862745098038


In [72]:
(best_xgb := xgb_grid_search.best_estimator_)
(y_pred := best_xgb.predict(X_test))
(y_prob := best_xgb.predict_proba(X_test)[:, 1])

{
    'accuracy': accuracy_score(y_test, y_pred),
    'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_prob)
}

{'accuracy': 0.8235294117647058,
 'balanced_accuracy': 0.625,
 'f1': 0.4,
 'precision': 1.0,
 'recall': 0.25,
 'roc_auc': 0.7692307692307693}

## <font color='red'> Exercise 3 </font>

Investigate which of these method provide a measure of feature importance and take a look at the results, where possible,

<font color='orange'>
The table below compares feature importance across models.

- AdaBoost, GradientBoosting, and XGBoost use built-in `feature_importances_`.
- HistGradientBoosting uses permutation importance.

Read the largest values in each column (or the top-summary table) to see which predictors were most important.
</font>

In [78]:
import pandas as pd

In [79]:
(feature_names := X_kyphosis.columns)

# Best fitted models already trained in this notebook
(ada_best := ada_grid_search.best_estimator_)
(gb_best := GB_grid_search.best_estimator_)
(hist_best := hist_GB_grid_search.best_estimator_)
(xgb_best := xgb_grid_search.best_estimator_)

# Built-in feature_importances_ (HistGradientBoostingClassifier does not expose this)
(ada_imp := pd.Series(ada_best.feature_importances_, index=feature_names, name='AdaBoost'))
(gb_imp := pd.Series(gb_best.feature_importances_, index=feature_names, name='GradientBoosting'))
(xgb_imp := pd.Series(xgb_best.feature_importances_, index=feature_names, name='XGBoost'))
(hist_imp := pd.Series([float('nan')] * len(feature_names), index=feature_names, name='HistGradientBoosting'))

(feature_importance_table := pd.concat([ada_imp, gb_imp, hist_imp, xgb_imp], axis=1).round(4))
feature_importance_table.sort_values('GradientBoosting', ascending=False)

,AdaBoost,GradientBoosting,HistGradientBoosting,XGBoost
Start,0.5494,0.5242,NaN,0.4932
Age,0.3197,0.3908,NaN,0.2104
Number,0.1309,0.0850,NaN,0.2964
